In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Folder link input
folder_path = '/content/drive/MyDrive/Train' # Replace 'Your_Folder_Name' with the actual folder name in your Drive

if os.path.exists(folder_path):
    print(f"Successfully accessed: {folder_path}")
    # List files to verify
    print("Files in folder:", os.listdir(folder_path))
else:
    print("Folder not found. Please ensure the folder name matches your Drive structure.")

Mounted at /content/drive
Successfully accessed: /content/drive/MyDrive/Train
Files in folder: ['Sylhet Train Translation.csv', 'Chittagong Train Translation.csv', 'Noakhali Train Translation.csv', 'Barishal Train Translation.csv', 'Mymensingh Train Translation.csv']


In [ ]:
import pandas as pd
import os
import glob

# Folder path (already mounted)
folder_path = '/content/drive/MyDrive/Train'

# সব CSV file নাও
files = glob.glob(os.path.join(folder_path, "*.csv"))

all_data = []

for file in files:
    print(f"Processing: {file}")

    df = pd.read_csv(file)

    # Column names clean করা
    df.columns = df.columns.str.strip()

    # Column names mapping
    region_col = "region_name"
    standard_col = "bangla_speech"
    english_col = "english_speech"

    # dialect column detect করা
    # ফাইল অনুযায়ী dialect কলামের নাম ভিন্ন হতে পারে (যেমন: sylhet_bangla_speech)
    # আমরা এমন কলাম খুঁজছি যা 'bangla_speech' নয় কিন্তু শেষে 'bangla_speech' আছে
    potential_dialect_cols = [c for c in df.columns if c.endswith('bangla_speech') and c != 'bangla_speech']

    if not potential_dialect_cols:
        print(f"No dialect column found in: {file}. Skipping...")
        continue

    dialect_col = potential_dialect_cols[0]

    # প্রয়োজনীয় কলামগুলো নিয়ে নতুন DataFrame তৈরি
    try:
        temp = df[[dialect_col, standard_col, english_col, region_col]].copy()

        temp.columns = [
            "regional_text",
            "standard_bangla",
            "english_text",
            "region"
        ]

        temp.dropna(inplace=True)
        all_data.append(temp)
    except KeyError as e:
        print(f"Column missing in {file}: {e}")

if all_data:
    # সব region একত্র করো
    final_dataset = pd.concat(all_data, ignore_index=True)

    # Save merged file
    output_path = "/content/drive/MyDrive/merged_parallel_dataset.csv"
    final_dataset.to_csv(output_path, index=False)

    print(f"\nTotal merged rows: {len(final_dataset)}")
    print(f"File saved at: {output_path}")
    display(final_dataset.head())
else:
    print("No data was merged.")

Processing: /content/drive/MyDrive/Train/Sylhet Train Translation.csv
Processing: /content/drive/MyDrive/Train/Chittagong Train Translation.csv
Processing: /content/drive/MyDrive/Train/Noakhali Train Translation.csv
Processing: /content/drive/MyDrive/Train/Barishal Train Translation.csv
Processing: /content/drive/MyDrive/Train/Mymensingh Train Translation.csv

Total merged rows: 9375
File saved at: /content/drive/MyDrive/merged_parallel_dataset.csv


,regional_text,standard_bangla,english_text,region
0,ভালা আছনি?,কেমন আছো ?,How are you?,Sylhet
1,আইজকু আমার মন ভালা নায়,আজকে আমার মন ভালো নেই,I'm not feeling well today,Sylhet
2,তুমি কিতা খরো?,তুমি কি করো ?,what are you doing,Sylhet
3,অউ গরমো আমার কুনতা ভালা লাগের না,এই গরমে আমার কিছু ভালো লাগে না,I don't like anything this summer,Sylhet
4,ফুয়াটায় এখটা সাদা রংগর শার্ট পিন্দিয়া আইছিল,ছেলেটি সাদা রঙয়ের একটি শার্ট পরে এসেছিল,The boy came wearing a white shirt,Sylhet


In [ ]:
import pandas as pd
import os
import glob



final_dataset = pd.read_csv("/content/drive/MyDrive/merged_parallel_dataset.csv")
print(final_dataset["region"].value_counts())


region
Sylhet        1875
Chittagong    1875
Noakhali      1875
Barishal      1875
Mymensingh    1875
Name: count, dtype: int64


In [ ]:
print(final_dataset.isnull().sum())

print("Total rows before:", len(final_dataset))

final_dataset = final_dataset.drop_duplicates()

print("Total rows after removing duplicates:", len(final_dataset))



regional_text      0
standard_bangla    0
english_text       0
region             0
dtype: int64
Total rows before: 9375
Total rows after removing duplicates: 9375


In [ ]:
final_dataset["regional_text"] = final_dataset["regional_text"].str.strip()
final_dataset["standard_bangla"] = final_dataset["standard_bangla"].str.strip()
final_dataset["english_text"] = final_dataset["english_text"].str.strip()


In [ ]:
final_dataset["dialect_len"] = final_dataset["regional_text"].apply(len)
final_dataset["standard_len"] = final_dataset["standard_bangla"].apply(len)

print(final_dataset[["dialect_len", "standard_len"]].describe())


       dialect_len  standard_len
count  9375.000000   9375.000000
mean     35.665067     34.911253
std      14.948451     14.642829
min       8.000000      9.000000
25%      25.000000     25.000000
50%      33.000000     32.000000
75%      43.000000     42.000000
max     113.000000    105.000000


In [ ]:
final_dataset = final_dataset[
    (final_dataset["regional_text"] != "") &
    (final_dataset["standard_bangla"] != "")
]


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    final_dataset,
    test_size=0.2,
    random_state=42,
    stratify=final_dataset["region"]
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

all_text = "".join(train_df["regional_text"].tolist()) + "".join(train_df["standard_bangla"].tolist())

vocab = sorted(set(all_text))

print("Total unique characters:", len(vocab))

special_tokens = ["<pad>", "<sos>", "<eos>"]

vocab = special_tokens + vocab

char2idx = {char: idx for idx, char in enumerate(vocab)}
idx2char = {idx: char for char, idx in char2idx.items()}

print("Vocabulary size:", len(vocab))

def encode_text(text, char2idx):
    return [char2idx["<sos>"]] + \
           [char2idx[char] for char in text if char in char2idx] + \
           [char2idx["<eos>"]]

sample = train_df.iloc[0]["regional_text"]
print(sample)
print(encode_text(sample, char2idx)[:20])


Train size: 7500
Test size: 1875
Total unique characters: 85
Vocabulary size: 88
এইডা কি ঠিক সিদ্ধান্ত অইবো আঙ্গো লাই?
[1, 24, 19, 40, 61, 3, 28, 62, 3, 39, 62, 28, 3, 58, 62, 45, 71, 46, 61, 47]


In [ ]:
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F

class DialectDataset(Dataset):
    def __init__(self, dataframe, char2idx):
        self.df = dataframe
        self.char2idx = char2idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        src = encode_text(self.df.iloc[idx]["regional_text"], self.char2idx)
        tgt = encode_text(self.df.iloc[idx]["standard_bangla"], self.char2idx)
        return torch.tensor(src), torch.tensor(tgt)

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_padded = pad_sequence(src_batch, padding_value=char2idx["<pad>"])
    tgt_padded = pad_sequence(tgt_batch, padding_value=char2idx["<pad>"])
    return src_padded, tgt_padded

train_dataset = DialectDataset(train_df, char2idx)
test_dataset = DialectDataset(test_df, char2idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.dropout = nn.Dropout(dropout)

        self.gru = nn.GRU(
            embed_size,
            hidden_size,
            bidirectional=True,
            dropout=dropout
        )

        self.fc = nn.Linear(hidden_size * 2, hidden_size)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))

        outputs, hidden = self.gru(embedded)

        # combine bidirectional hidden
        hidden = torch.tanh(
            self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        ).unsqueeze(0)

        return outputs, hidden



class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()

        self.attn = nn.Linear(hidden_size * 3, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.shape[0]

        hidden = hidden.repeat(seq_len, 1, 1)

        energy = torch.tanh(
            self.attn(torch.cat((hidden, encoder_outputs), dim=2))
        )

        attention = self.v(energy).squeeze(2)

        return torch.softmax(attention, dim=0)


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.dropout = nn.Dropout(dropout)

        self.attention = Attention(hidden_size)

        self.gru = nn.GRU(
            embed_size + hidden_size * 2,
            hidden_size,
            dropout=dropout
        )

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden, encoder_outputs):
        x = x.unsqueeze(0)

        embedded = self.dropout(self.embedding(x))

        attn_weights = self.attention(hidden, encoder_outputs).unsqueeze(2)

        context = torch.sum(attn_weights * encoder_outputs, dim=0).unsqueeze(0)

        rnn_input = torch.cat((embedded, context), dim=2)

        output, hidden = self.gru(rnn_input, hidden)

        prediction = self.fc(output.squeeze(0))

        return prediction, hidden


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.3):
        batch_size = tgt.shape[1]
        tgt_len = tgt.shape[0]
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(tgt_len, batch_size, vocab_size).to(self.device)

        encoder_outputs, hidden = self.encoder(src)

        input = tgt[0]

        for t in range(1, tgt_len):
            output, hidden = self.decoder(input, hidden, encoder_outputs)

            outputs[t] = output

            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)

            input = tgt[t] if teacher_force else top1

        return outputs


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab_size = len(vocab)
embed_size = 128
hidden_size = 256
encoder = Encoder(vocab_size, embed_size, hidden_size)
decoder = Decoder(vocab_size, embed_size, hidden_size)
model = Seq2Seq(encoder, decoder, device).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss(ignore_index=char2idx["<pad>"])

def train(model, loader, epoch):
    model.train()
    total_loss = 0

    teacher_forcing_ratio = max(0.1, 0.5 - epoch * 0.02)

    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)

        optimizer.zero_grad()

        output = model(src, tgt, teacher_forcing_ratio)

        output_dim = output.shape[-1]

        output = output[1:].reshape(-1, output_dim)
        tgt = tgt[1:].reshape(-1)

        loss = criterion(output, tgt)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            output = model(src, tgt, teacher_forcing_ratio=0)
            output_dim = output.shape[-1]
            output = output[1:].reshape(-1, output_dim)
            tgt = tgt[1:].reshape(-1)
            loss = criterion(output, tgt)
            total_loss += loss.item()
    return total_loss / len(loader)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn(


In [ ]:
best_val_loss = float('inf')
patience = 5
counter = 0

for epoch in range(30):

    train_loss = train(model, train_loader, epoch)
    val_loss = evaluate(model, test_loader)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered")
            break


Epoch 1
Train Loss: 3.0061
Val Loss: 3.1930
Epoch 2
Train Loss: 2.6213
Val Loss: 2.9399
Epoch 3
Train Loss: 2.1821
Val Loss: 2.7089
Epoch 4
Train Loss: 1.9083
Val Loss: 2.6053
Epoch 5
Train Loss: 1.7169
Val Loss: 2.5171
Epoch 6
Train Loss: 1.5918
Val Loss: 2.5341
Epoch 7
Train Loss: 1.5244
Val Loss: 2.4370
Epoch 8
Train Loss: 1.4579
Val Loss: 2.4840
Epoch 9
Train Loss: 1.4377
Val Loss: 2.4493
Epoch 10
Train Loss: 1.3864
Val Loss: 2.4111
Epoch 11
Train Loss: 1.3582
Val Loss: 2.3016
Epoch 12
Train Loss: 1.3441
Val Loss: 2.2514
Epoch 13
Train Loss: 1.3336
Val Loss: 2.3405
Epoch 14
Train Loss: 1.3239
Val Loss: 2.2031
Epoch 15
Train Loss: 1.3534
Val Loss: 2.1159
Epoch 16
Train Loss: 1.3180
Val Loss: 2.1777
Epoch 17
Train Loss: 1.3699
Val Loss: 2.1656
Epoch 18
Train Loss: 1.3739
Val Loss: 2.0145
Epoch 19
Train Loss: 1.3782
Val Loss: 2.0691
Epoch 20
Train Loss: 1.4135
Val Loss: 2.0521
Epoch 21
Train Loss: 1.4183
Val Loss: 1.9933
Epoch 22
Train Loss: 1.3900
Val Loss: 1.9501
Epoch 23
Train Loss

In [ ]:
def translate_sentence(model, sentence):
    model.eval()
    with torch.no_grad():
        encoded = torch.tensor(encode_text(sentence, char2idx)).unsqueeze(1).to(device)
        encoder_outputs, hidden = model.encoder(encoded)
        input = torch.tensor([char2idx["<sos>"]]).to(device)
        result = []

        for _ in range(100):
            output, hidden = model.decoder(input, hidden, encoder_outputs)
            top1 = output.argmax(1)
            char = idx2char[top1.item()]
            if char == "<eos>":
                break
            result.append(char)
            input = top1

    return "".join(result)

print(translate_sentence(model, "অ্যাঁর হথা বুঝস তুঁই"))

print(translate_sentence(model,"আইজ হগ বুজস তুই"
))

তার পথা বুঝে  ুু তুমি
আজ সা বুজজ তুমি 


In [ ]:
!pip install sentencepiece -q


import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import random
import sentencepiece as spm
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load dataset
final_dataset = pd.read_csv("/content/drive/MyDrive/merged_parallel_dataset.csv")
final_dataset = final_dataset.dropna(subset=["regional_text", "standard_bangla"])
print(final_dataset.head())
print(final_dataset["region"].value_counts())

# Save training text for tokenizer (both sides)
os.makedirs("tok_data", exist_ok=True)
with open("tok_data/regional.txt", "w", encoding="utf-8") as f_in, \
     open("tok_data/standard.txt", "w", encoding="utf-8") as f_out:
    for _, row in final_dataset.iterrows():
        f_in.write(str(row["regional_text"]).strip() + "\n")
        f_out.write(str(row["standard_bangla"]).strip() + "\n")

# Train two separate tokenizers (you can also share one vocab)
vocab_size = 8000  # tune if needed

spm.SentencePieceTrainer.Train(
    input="tok_data/regional.txt",
    model_prefix="sp_regional",
    vocab_size=vocab_size,
    character_coverage=0.9995,
    model_type="bpe"
)

spm.SentencePieceTrainer.Train(
    input="tok_data/standard.txt",
    model_prefix="sp_standard",
    vocab_size=vocab_size,
    character_coverage=0.9995,
    model_type="bpe"
)

sp_src = spm.SentencePieceProcessor(model_file="sp_regional.model")
sp_tgt = spm.SentencePieceProcessor(model_file="sp_standard.model")

# Define special tokens
PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2


Device: cuda
                                 regional_text  \
0                                   ভালা আছনি?   
1                       আইজকু আমার মন ভালা নায়   
2                               তুমি কিতা খরো?   
3             অউ গরমো আমার কুনতা ভালা লাগের না   
4  ফুয়াটায় এখটা সাদা রংগর শার্ট পিন্দিয়া আইছিল   

                           standard_bangla  \
0                               কেমন আছো ?   
1                    আজকে আমার মন ভালো নেই   
2                            তুমি কি করো ?   
3           এই গরমে আমার কিছু ভালো লাগে না   
4  ছেলেটি সাদা রঙয়ের একটি শার্ট পরে এসেছিল   

                         english_text  region  
0                        How are you?  Sylhet  
1          I'm not feeling well today  Sylhet  
2                  what are you doing  Sylhet  
3   I don't like anything this summer  Sylhet  
4  The boy came wearing a white shirt  Sylhet  
region
Sylhet        1875
Chittagong    1875
Noakhali      1875
Barishal      1875
Mymensingh    1875
Name: count, dtype:

In [ ]:
class BanglaNormDataset(Dataset):
    def __init__(self, df, sp_src, sp_tgt, max_len=64):
        self.df = df.reset_index(drop=True)
        self.sp_src = sp_src
        self.sp_tgt = sp_tgt
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        src_text = str(self.df.loc[idx, "regional_text"]).strip()
        tgt_text = str(self.df.loc[idx, "standard_bangla"]).strip()

        src_ids = [BOS_IDX] + self.sp_src.encode(src_text, out_type=int)[: self.max_len-2] + [EOS_IDX]
        tgt_ids = [BOS_IDX] + self.sp_tgt.encode(tgt_text, out_type=int)[: self.max_len-2] + [EOS_IDX]

        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX, batch_first=True)
    return src_batch, tgt_batch

# Train/val split
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(final_dataset, test_size=0.1, random_state=42)

train_dataset = BanglaNormDataset(train_df, sp_src, sp_tgt)
val_dataset   = BanglaNormDataset(val_df,   sp_src, sp_tgt)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, collate_fn=collate_fn)

class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        num_encoder_layers,
        num_decoder_layers,
        emb_size,
        nhead,
        src_vocab_size,
        tgt_vocab_size,
        dim_feedforward=2048,
        dropout=0.1,
    ):
        super().__init__()

        self.src_tok_emb = nn.Embedding(src_vocab_size, emb_size, padding_idx=PAD_IDX)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, emb_size, padding_idx=PAD_IDX)
        self.positional_encoding = PositionalEncoding(emb_size, dropout=dropout)

        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.generator = nn.Linear(emb_size, tgt_vocab_size)

    def forward(self, src, tgt):
        # src, tgt: (batch, seq_len)
        src_mask = None
        tgt_mask = self.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)

        src_key_padding_mask = (src == PAD_IDX)
        tgt_key_padding_mask = (tgt == PAD_IDX)

        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt))

        outs = self.transformer(
            src_emb,
            tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )
        return self.generator(outs)

    @staticmethod
    def generate_square_subsequent_mask(sz):
        mask = torch.triu(torch.ones(sz, sz) * float("-inf"), diagonal=1)
        return mask

class PositionalEncoding(nn.Module):
    def __init__(self, emb_size, dropout=0.1, maxlen=5000):
        super().__init__()
        den = torch.exp(-torch.arange(0, emb_size, 2) * (torch.log(torch.tensor(10000.0)) / emb_size))
        pos = torch.arange(0, maxlen).reshape(maxlen, 1)
        pos_embedding = torch.zeros((maxlen, emb_size))
        pos_embedding[:, 0::2] = torch.sin(pos * den)
        pos_embedding[:, 1::2] = torch.cos(pos * den)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("pos_embedding", pos_embedding)

    def forward(self, token_embedding):
        return self.dropout(token_embedding + self.pos_embedding[: token_embedding.size(1), :])



In [ ]:
SRC_VOCAB_SIZE = sp_src.vocab_size()
TGT_VOCAB_SIZE = sp_tgt.vocab_size()

model = Seq2SeqTransformer(
    num_encoder_layers=3,
    num_decoder_layers=3,
    emb_size=256,
    nhead=8,
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    dim_feedforward=512,
    dropout=0.1,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

def train_epoch(model, loader):
    model.train()
    total_loss = 0
    for src, tgt in loader:
        src = src.to(device)
        tgt = tgt.to(device)

        # Decoder input = everything except last token
        tgt_input = tgt[:, :-1]
        # Target for loss = everything except first token
        tgt_out   = tgt[:, 1:].contiguous().view(-1)

        logits = model(src, tgt_input)  # (batch, seq_len-1, vocab)
        logits = logits.view(-1, logits.size(-1))

        loss = criterion(logits, tgt_out)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0
    for src, tgt in loader:
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_out   = tgt[:, 1:].contiguous().view(-1)
        logits = model(src, tgt_input)
        logits = logits.view(-1, logits.size(-1))
        loss = criterion(logits, tgt_out)
        total_loss += loss.item()
    return total_loss / len(loader)

EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader)
    val_loss   = evaluate(model, val_loader)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")



/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1: train_loss=6.5372, val_loss=5.8664
Epoch 2: train_loss=5.5293, val_loss=5.1427
Epoch 3: train_loss=4.8291, val_loss=4.4603
Epoch 4: train_loss=4.1587, val_loss=3.7907
Epoch 5: train_loss=3.5196, val_loss=3.1956
Epoch 6: train_loss=2.9163, val_loss=2.6028
Epoch 7: train_loss=2.3492, val_loss=2.0968
Epoch 8: train_loss=1.8329, val_loss=1.6291
Epoch 9: train_loss=1.3958, val_loss=1.2489
Epoch 10: train_loss=1.0468, val_loss=0.9533


In [ ]:
EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader)
    val_loss   = evaluate(model, val_loader)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

Epoch 1: train_loss=0.7800, val_loss=0.7269
Epoch 2: train_loss=0.5896, val_loss=0.5854
Epoch 3: train_loss=0.4524, val_loss=0.4978
Epoch 4: train_loss=0.3511, val_loss=0.4397
Epoch 5: train_loss=0.2781, val_loss=0.3935
Epoch 6: train_loss=0.2213, val_loss=0.3700
Epoch 7: train_loss=0.1814, val_loss=0.3453
Epoch 8: train_loss=0.1504, val_loss=0.3263
Epoch 9: train_loss=0.1277, val_loss=0.3231
Epoch 10: train_loss=0.1089, val_loss=0.3142


In [ ]:
EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader)
    val_loss   = evaluate(model, val_loader)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

Epoch 1: train_loss=0.0899, val_loss=0.3052
Epoch 2: train_loss=0.0791, val_loss=0.3038
Epoch 3: train_loss=0.0692, val_loss=0.2974
Epoch 4: train_loss=0.0649, val_loss=0.3076
Epoch 5: train_loss=0.0594, val_loss=0.2967
Epoch 6: train_loss=0.0539, val_loss=0.2949
Epoch 7: train_loss=0.0479, val_loss=0.3015
Epoch 8: train_loss=0.0450, val_loss=0.2967
Epoch 9: train_loss=0.0430, val_loss=0.2888
Epoch 10: train_loss=0.0394, val_loss=0.2884


In [ ]:
EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader)
    val_loss   = evaluate(model, val_loader)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

Epoch 1: train_loss=0.0348, val_loss=0.2902
Epoch 2: train_loss=0.0350, val_loss=0.2952
Epoch 3: train_loss=0.0349, val_loss=0.2962
Epoch 4: train_loss=0.0330, val_loss=0.2976
Epoch 5: train_loss=0.0325, val_loss=0.3053
Epoch 6: train_loss=0.0314, val_loss=0.3022
Epoch 7: train_loss=0.0278, val_loss=0.3018
Epoch 8: train_loss=0.0281, val_loss=0.3044
Epoch 9: train_loss=0.0263, val_loss=0.3078
Epoch 10: train_loss=0.0261, val_loss=0.3008


In [ ]:
@torch.no_grad()
def translate_sentence(text):
    model.eval()
    # Encode source
    src_ids = [BOS_IDX] + sp_src.encode(text, out_type=int) + [EOS_IDX]
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)  # (1, seq)

    # Start decoder with BOS
    tgt_ids = [BOS_IDX]
    max_len = 64

    for _ in range(max_len):
        tgt = torch.tensor(tgt_ids, dtype=torch.long).unsqueeze(0).to(device)
        logits = model(src, tgt)          # (1, len, vocab)
        next_token_logits = logits[0, -1] # last step
        next_token = next_token_logits.argmax(-1).item()

        if next_token == EOS_IDX:
            break
        tgt_ids.append(next_token)

    # Remove BOS
    out_subwords = sp_tgt.decode(tgt_ids[1:])
    return out_subwords

# Test on some sample rows
for i in range(10):
    regional = final_dataset.iloc[i]["regional_text"]
    standard = final_dataset.iloc[i]["standard_bangla"]
    pred = translate_sentence(regional)
    print(f"\nRegional : {regional}")
    print(f"Gold     : {standard}")
    print(f"Pred     : {pred}")



Regional : ভালা আছনি?
Gold     : কেমন আছো ?
Pred     : কেমন আছো ?

Regional : আইজকু আমার মন ভালা নায়
Gold     : আজকে আমার মন ভালো নেই
Pred     : আজকে আমার মন ভালো নেই

Regional : তুমি কিতা খরো?
Gold     : তুমি কি করো ?
Pred     : তুমি কি করো ?

Regional : অউ গরমো আমার কুনতা ভালা লাগের না
Gold     : এই গরমে আমার কিছু ভালো লাগে না
Pred     : এই গরমে আমার কিছু ভালো লাগে না

Regional : ফুয়াটায় এখটা সাদা রংগর শার্ট পিন্দিয়া আইছিল
Gold     : ছেলেটি সাদা রঙয়ের একটি শার্ট পরে এসেছিল
Pred     : ছেলেটি সাদা রঙয়ের একটি শার্ট পরে এসেছিল

Regional : ফুড়িটায় লাল রংগর শাড়ি পিন্দিয়া মোর লগে দেখা করাত আইছিল
Gold     : মেয়েটি লাল রঙয়ের শাড়ি পরে আমার সাথে দেখা করতে এসেছিল
Pred     : মেয়েটি লাল রঙয়ের শাড়ি পরে আমার সাথে দেখা করতে এসেছিল

Regional : ফুয়াটায় সিলেট থাকি ঢাকাত আইছে
Gold     : ছেলেটি সিলেট থেকে ঢাকায় এসেছে
Pred     : ছেলেটি সিলেট থেকে ঢাকায় এসেছে

Regional : ফুড়িটায় সিলেট থাকি আওয়া অউ ফুয়াটারে অনেক ভালা পায়
Gold     : মেয়েটি সিলেট থেকে আসা এই ছেলেটিকে অনেক ভালবাসে
Pred     : মেয়েটি সিলেট

In [ ]:
!pip install sacrebleu -q

import sacrebleu
from tqdm import tqdm

model.eval()

preds = []
refs = []

for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
    src = str(row["regional_text"])
    tgt = str(row["standard_bangla"])

    pred = translate_sentence(src)  # uses your trained model
    preds.append(pred)
    refs.append(tgt)

refs_list = [refs]  # shape: [num_refs][num_sentences]

# Corpus BLEU
bleu = sacrebleu.corpus_bleu(preds, refs_list)   # default: BLEU-4
print(f"Corpus BLEU: {bleu.score:.2f}")

# chrF++ (character F-score, strong for normalization tasks)
chrf = sacrebleu.corpus_chrf(preds, refs_list)   # default chrF++
print(f"Corpus chrF++: {chrf.score:.2f}")



100%|██████████| 938/938 [00:51<00:00, 18.04it/s]


Corpus BLEU: 58.33
Corpus chrF++: 71.01


In [ ]:
@torch.no_grad()
def exact_match_accuracy_transformer(df, max_samples=None):
    model.eval()
    if max_samples is not None and max_samples < len(df):
        eval_df = df.sample(n=max_samples, random_state=42)
    else:
        eval_df = df

    total = 0
    correct = 0

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        src = str(row["regional_text"])
        tgt = str(row["standard_bangla"]).strip()

        pred = translate_sentence(src).strip()  # Transformer version

        total += 1
        if pred == tgt:
            correct += 1

    return 100.0 * correct / total

acc_trans = exact_match_accuracy_transformer(val_df)
print(f"Transformer exact-match accuracy on validation set: {acc_trans:.2f}%")


  0%|          | 0/938 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
100%|██████████| 938/938 [00:52<00:00, 17.99it/s]

Transformer exact-match accuracy on validation set: 38.38%


# **Bi-LSTM**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EMB_SIZE = 256
HID_SIZE = 256
NUM_LAYERS = 2
DROPOUT = 0.3
TEACHER_FORCING_RATIO = 0.6  # 0.5–0.7

class EncoderBiLSTM(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout, pad_idx):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=pad_idx)
        self.rnn = nn.LSTM(
            emb_dim,
            hid_dim,
            num_layers=n_layers,
            bidirectional=True,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        # Combine the two directions for initial decoder state
        self.fc_h = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_c = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, src, src_lengths=None):
        # src: (batch, src_len)
        embedded = self.dropout(self.embedding(src))  # (batch, src_len, emb_dim)

        # If you have packed sequences, you can pack here; for now, assume padded.
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs: (batch, src_len, hid_dim * 2) because bidirectional
        # hidden: (num_layers*2, batch, hid_dim)

        # Take last layer's forward/backward hidden states and map to decoder size
        # hidden[-2,:,:] = last forward, hidden[-1,:,:] = last backward
        h_forward = hidden[-2,:,:]
        h_backward = hidden[-1,:,:]
        h_cat = torch.cat((h_forward, h_backward), dim=1)  # (batch, 2*hid_dim)
        dec_hidden = torch.tanh(self.fc_h(h_cat)).unsqueeze(0).repeat(self.n_layers, 1, 1)
        dec_cell   = torch.tanh(self.fc_c(h_cat)).unsqueeze(0).repeat(self.n_layers, 1, 1)

        return outputs, dec_hidden, dec_cell
class BahdanauAttention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        # enc_hid_dim * 2 because BiLSTM
        self.attn = nn.Linear(enc_hid_dim * 2 + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, src_mask):
        # decoder_hidden: (num_layers, batch, dec_hid_dim) -> use last layer
        # encoder_outputs: (batch, src_len, enc_hid_dim*2)
        # src_mask: (batch, src_len) (True where PAD)

        dec_hidden = decoder_hidden[-1].unsqueeze(1)  # (batch, 1, dec_hid_dim)
        src_len = encoder_outputs.size(1)

        dec_hidden_repeat = dec_hidden.repeat(1, src_len, 1)  # (batch, src_len, dec_hid_dim)
        energy = torch.tanh(
            self.attn(torch.cat((dec_hidden_repeat, encoder_outputs), dim=2))
        )  # (batch, src_len, dec_hid_dim)

        attn_scores = self.v(energy).squeeze(-1)  # (batch, src_len)

        # Mask out PAD positions
        attn_scores = attn_scores.masked_fill(src_mask, float("-inf"))
        attn_weights = F.softmax(attn_scores, dim=1)  # (batch, src_len)

        return attn_weights

class DecoderLSTMWithAttention(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim,
                 n_layers, dropout, pad_idx):
        super().__init__()
        self.output_dim = output_dim
        self.emb_dim = emb_dim
        self.dec_hid_dim = dec_hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=pad_idx)
        self.attention = BahdanauAttention(enc_hid_dim, dec_hid_dim)

        self.rnn = nn.LSTM(
            emb_dim + enc_hid_dim * 2,
            dec_hid_dim,
            num_layers=n_layers,
            batch_first=True,
        )
        self.fc_out = nn.Linear(enc_hid_dim * 2 + dec_hid_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_tok, hidden, cell, encoder_outputs, src_mask):
        # input_tok: (batch,) current token ids
        input_tok = input_tok.unsqueeze(1)  # (batch, 1)
        embedded = self.dropout(self.embedding(input_tok))  # (batch, 1, emb_dim)

        attn_weights = self.attention(hidden, encoder_outputs, src_mask)  # (batch, src_len)
        attn_weights = attn_weights.unsqueeze(1)  # (batch, 1, src_len)

        context = torch.bmm(attn_weights, encoder_outputs)  # (batch, 1, enc_hid_dim*2)

        rnn_input = torch.cat((embedded, context), dim=2)  # (batch, 1, emb_dim+enc_hid_dim*2)

        outputs, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        # outputs: (batch, 1, dec_hid_dim)

        output_cat = torch.cat((outputs, context, embedded), dim=2)  # (batch, 1, dec_hid_dim+enc_hid_dim*2+emb_dim)

        prediction = self.fc_out(output_cat).squeeze(1)  # (batch, output_dim)

        return prediction, hidden, cell, attn_weights.squeeze(1)
class Seq2SeqBiLSTM(nn.Module):
    def __init__(self, encoder, decoder, pad_idx):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_idx = pad_idx

    def create_src_mask(self, src):
        # src: (batch, src_len)
        return (src == self.pad_idx)  # True where PAD

    def forward(self, src, tgt, teacher_forcing_ratio=TEACHER_FORCING_RATIO):
        # src: (batch, src_len)
        # tgt: (batch, tgt_len)
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        tgt_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size, device=src.device)

        encoder_outputs, hidden, cell = self.encoder(src)
        src_mask = self.create_src_mask(src)

        # first input to decoder is BOS token for all
        input_tok = tgt[:, 0]  # BOS

        for t in range(1, tgt_len):
            output, hidden, cell, _ = self.decoder(
                input_tok, hidden, cell, encoder_outputs, src_mask
            )
            outputs[:, t] = output

            # decide if teacher forcing
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)  # (batch,)

            input_tok = tgt[:, t] if teacher_force else top1

        return outputs


In [ ]:
encoder = EncoderBiLSTM(
    input_dim=SRC_VOCAB_SIZE,
    emb_dim=EMB_SIZE,
    hid_dim=HID_SIZE,
    n_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_idx=PAD_IDX,
)

decoder = DecoderLSTMWithAttention(
    output_dim=TGT_VOCAB_SIZE,
    emb_dim=EMB_SIZE,
    enc_hid_dim=HID_SIZE,
    dec_hid_dim=HID_SIZE,
    n_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_idx=PAD_IDX,
)

model_bilstm = Seq2SeqBiLSTM(encoder, decoder, PAD_IDX).to(device)

optimizer = torch.optim.Adam(model_bilstm.parameters(), lr=1e-3)  # 1e-3–2e-3
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

def train_epoch_bilstm(model, loader):
    model.train()
    total_loss = 0

    for src, tgt in loader:
        src = src.to(device)
        tgt = tgt.to(device)

        optimizer.zero_grad()

        # outputs: (batch, tgt_len, vocab_size)
        outputs = model(src, tgt)
        # ignore t=0 (BOS position)
        output_dim = outputs.size(-1)
        logits = outputs[:, 1:].contiguous().view(-1, output_dim)
        tgt_out = tgt[:, 1:].contiguous().view(-1)

        loss = criterion(logits, tgt_out)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

@torch.no_grad()
def eval_epoch_bilstm(model, loader):
    model.eval()
    total_loss = 0

    for src, tgt in loader:
        src = src.to(device)
        tgt = tgt.to(device)

        outputs = model(src, tgt, teacher_forcing_ratio=0.0)  # no TF for val
        output_dim = outputs.size(-1)
        logits = outputs[:, 1:].contiguous().view(-1, output_dim)
        tgt_out = tgt[:, 1:].contiguous().view(-1)

        loss = criterion(logits, tgt_out)
        total_loss += loss.item()

    return total_loss / len(loader)

EPOCHS = 15

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch_bilstm(model_bilstm, train_loader)
    val_loss = eval_epoch_bilstm(model_bilstm, val_loader)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")


Epoch 1: train_loss=6.1578, val_loss=5.8376
Epoch 2: train_loss=4.5578, val_loss=4.8470
Epoch 3: train_loss=3.0068, val_loss=3.6572
Epoch 4: train_loss=1.8064, val_loss=2.7525
Epoch 5: train_loss=1.1028, val_loss=2.1926
Epoch 6: train_loss=0.6816, val_loss=1.8467
Epoch 7: train_loss=0.4297, val_loss=1.7466
Epoch 8: train_loss=0.3056, val_loss=1.5756
Epoch 9: train_loss=0.2110, val_loss=1.4641
Epoch 10: train_loss=0.1526, val_loss=1.4420
Epoch 11: train_loss=0.1166, val_loss=1.4457
Epoch 12: train_loss=0.0923, val_loss=1.4196
Epoch 13: train_loss=0.0735, val_loss=1.4376
Epoch 14: train_loss=0.0608, val_loss=1.4097
Epoch 15: train_loss=0.0500, val_loss=1.3679


In [ ]:
EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch_bilstm(model_bilstm, train_loader)
    val_loss = eval_epoch_bilstm(model_bilstm, val_loader)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

Epoch 1: train_loss=0.0428, val_loss=1.3816
Epoch 2: train_loss=0.0380, val_loss=1.3866
Epoch 3: train_loss=0.0382, val_loss=1.3902
Epoch 4: train_loss=0.0297, val_loss=1.4238
Epoch 5: train_loss=0.0283, val_loss=1.3262
Epoch 6: train_loss=0.0243, val_loss=1.3387
Epoch 7: train_loss=0.0238, val_loss=1.4033
Epoch 8: train_loss=0.0250, val_loss=1.3966
Epoch 9: train_loss=0.0206, val_loss=1.4676
Epoch 10: train_loss=0.0163, val_loss=1.4162


In [ ]:
@torch.no_grad()
def translate_bilstm(text, sp_src, sp_tgt, max_len=64):
    model_bilstm.eval()
    src_ids = [BOS_IDX] + sp_src.encode(text, out_type=int) + [EOS_IDX]
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)

    encoder_outputs, hidden, cell = model_bilstm.encoder(src)
    src_mask = model_bilstm.create_src_mask(src)

    input_tok = torch.tensor([BOS_IDX], device=device)
    generated = [BOS_IDX]

    for _ in range(max_len):
        output, hidden, cell, _ = model_bilstm.decoder(
            input_tok, hidden, cell, encoder_outputs, src_mask
        )
        top1 = output.argmax(1).item()
        if top1 == EOS_IDX:
            break
        generated.append(top1)
        input_tok = torch.tensor([top1], device=device)

    decoded = sp_tgt.decode(generated[1:])
    return decoded

# quick sanity check
for i in range(5):
    regional = final_dataset.iloc[i]["regional_text"]
    gold = final_dataset.iloc[i]["standard_bangla"]
    pred = translate_bilstm(regional, sp_src, sp_tgt)
    print("\nSRC :", regional)
    print("GOLD:", gold)
    print("PRED:", pred)



SRC : ভালা আছনি?
GOLD: কেমন আছো ?
PRED: কেমন কেমন আছো?

SRC : আইজকু আমার মন ভালা নায়
GOLD: আজকে আমার মন ভালো নেই
PRED: আজকে আমার মন ভালো নেই

SRC : তুমি কিতা খরো?
GOLD: তুমি কি করো ?
PRED: তুমি কি করো ?

SRC : অউ গরমো আমার কুনতা ভালা লাগের না
GOLD: এই গরমে আমার কিছু ভালো লাগে না
PRED: এই গরমে আমার কিছু ভালো লাগে না

SRC : ফুয়াটায় এখটা সাদা রংগর শার্ট পিন্দিয়া আইছিল
GOLD: ছেলেটি সাদা রঙয়ের একটি শার্ট পরে এসেছিল
PRED: ছেলেটি সাদা রঙয়ের একটি শার্ট পরে এসেছিল


In [ ]:
import sacrebleu
from tqdm import tqdm

@torch.no_grad()
def evaluate_bilstm_bleu_chrf(model, df, sp_src, sp_tgt, max_samples=None):
    model.eval()
    preds = []
    refs = []

    if max_samples is not None and max_samples < len(df):
        eval_df = df.sample(n=max_samples, random_state=42)
    else:
        eval_df = df

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        src = str(row["regional_text"])
        tgt = str(row["standard_bangla"])

        pred = translate_bilstm(src, sp_src, sp_tgt)
        preds.append(pred)
        refs.append(tgt)

    # sacrebleu expects references as list-of-lists
    refs_list = [refs]

    bleu = sacrebleu.corpus_bleu(preds, refs_list).score
    chrf = sacrebleu.corpus_chrf(preds, refs_list).score

    return bleu, chrf

bilstm_bleu, bilstm_chrf = evaluate_bilstm_bleu_chrf(
    model_bilstm, val_df, sp_src, sp_tgt, max_samples=None
)

print(f"BiLSTM: BLEU = {bilstm_bleu:.2f}, chrF++ = {bilstm_chrf:.2f}")


100%|██████████| 938/938 [00:09<00:00, 95.30it/s] 


BiLSTM: BLEU = 60.55, chrF++ = 74.93


In [ ]:
from tqdm import tqdm

@torch.no_grad()
def exact_match_accuracy_bilstm(model, df, sp_src, sp_tgt, max_samples=None):
    model.eval()
    if max_samples is not None and max_samples < len(df):
        eval_df = df.sample(n=max_samples, random_state=42)
    else:
        eval_df = df

    total = 0
    correct = 0

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        src = str(row["regional_text"])
        tgt = str(row["standard_bangla"]).strip()

        pred = translate_bilstm(src, sp_src, sp_tgt).strip()

        total += 1
        if pred == tgt:
            correct += 1

    acc = 100.0 * correct / total
    return acc

acc_bilstm = exact_match_accuracy_bilstm(model_bilstm, val_df, sp_src, sp_tgt)
print(f"BiLSTM exact-match accuracy on validation set: {acc_bilstm:.2f}%")


100%|██████████| 938/938 [00:11<00:00, 83.43it/s] 

BiLSTM exact-match accuracy on validation set: 37.21%


# **Bi-GRU**

In [ ]:
!pip install sentencepiece -q


import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import random
import sentencepiece as spm
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load dataset
final_dataset = pd.read_csv("/content/drive/MyDrive/merged_parallel_dataset.csv")
final_dataset = final_dataset.dropna(subset=["regional_text", "standard_bangla"])
print(final_dataset.head())
print(final_dataset["region"].value_counts())

# Save training text for tokenizer (both sides)
os.makedirs("tok_data", exist_ok=True)
with open("tok_data/regional.txt", "w", encoding="utf-8") as f_in, \
     open("tok_data/standard.txt", "w", encoding="utf-8") as f_out:
    for _, row in final_dataset.iterrows():
        f_in.write(str(row["regional_text"]).strip() + "\n")
        f_out.write(str(row["standard_bangla"]).strip() + "\n")

# Train two separate tokenizers (you can also share one vocab)
vocab_size = 8000  # tune if needed

spm.SentencePieceTrainer.Train(
    input="tok_data/regional.txt",
    model_prefix="sp_regional",
    vocab_size=vocab_size,
    character_coverage=0.9995,
    model_type="bpe"
)

spm.SentencePieceTrainer.Train(
    input="tok_data/standard.txt",
    model_prefix="sp_standard",
    vocab_size=vocab_size,
    character_coverage=0.9995,
    model_type="bpe"
)

sp_src = spm.SentencePieceProcessor(model_file="sp_regional.model")
sp_tgt = spm.SentencePieceProcessor(model_file="sp_standard.model")

# Define special tokens
PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2



Device: cuda
                                 regional_text  \
0                                   ভালা আছনি?   
1                       আইজকু আমার মন ভালা নায়   
2                               তুমি কিতা খরো?   
3             অউ গরমো আমার কুনতা ভালা লাগের না   
4  ফুয়াটায় এখটা সাদা রংগর শার্ট পিন্দিয়া আইছিল   

                           standard_bangla  \
0                               কেমন আছো ?   
1                    আজকে আমার মন ভালো নেই   
2                            তুমি কি করো ?   
3           এই গরমে আমার কিছু ভালো লাগে না   
4  ছেলেটি সাদা রঙয়ের একটি শার্ট পরে এসেছিল   

                         english_text  region  
0                        How are you?  Sylhet  
1          I'm not feeling well today  Sylhet  
2                  what are you doing  Sylhet  
3   I don't like anything this summer  Sylhet  
4  The boy came wearing a white shirt  Sylhet  
region
Sylhet        1875
Chittagong    1875
Noakhali      1875
Barishal      1875
Mymensingh    1875
Name: count, dtype:

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

MAX_LEN = 64  # you can tune

class BanglaNormDataset(Dataset):
    def __init__(self, df, sp_src, sp_tgt, max_len=MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.sp_src = sp_src
        self.sp_tgt = sp_tgt
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        src_text = str(self.df.loc[idx, "regional_text"]).strip()
        tgt_text = str(self.df.loc[idx, "standard_bangla"]).strip()

        src_ids = [BOS_IDX] + self.sp_src.encode(src_text, out_type=int)[: self.max_len-2] + [EOS_IDX]
        tgt_ids = [BOS_IDX] + self.sp_tgt.encode(tgt_text, out_type=int)[: self.max_len-2] + [EOS_IDX]

        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    tgt_batch = pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_IDX)
    return src_batch, tgt_batch

from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(final_dataset, test_size=0.1, random_state=42)

train_dataset = BanglaNormDataset(train_df, sp_src, sp_tgt)
val_dataset   = BanglaNormDataset(val_df,   sp_src, sp_tgt)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, collate_fn=collate_fn)

SRC_VOCAB_SIZE = sp_src.vocab_size()
TGT_VOCAB_SIZE = sp_tgt.vocab_size()


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import random

EMB_SIZE  = 256
HID_SIZE  = 256
NUM_LAYERS = 2
DROPOUT  = 0.3
TEACHER_FORCING_RATIO = 0.6

class EncoderBiGRU(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout, pad_idx):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(
            emb_dim,
            hid_dim,
            num_layers=n_layers,
            bidirectional=True,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc_h = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, src):
        # src: (batch, src_len)
        embedded = self.dropout(self.embedding(src))  # (batch, src_len, emb_dim)
        outputs, hidden = self.rnn(embedded)
        # outputs: (batch, src_len, 2*hid_dim)
        # hidden: (2*n_layers, batch, hid_dim)

        # last forward & backward states of top layer
        h_forward = hidden[-2,:,:]
        h_backward = hidden[-1,:,:]
        h_cat = torch.cat((h_forward, h_backward), dim=1)  # (batch, 2*hid_dim)
        dec_hidden = torch.tanh(self.fc_h(h_cat)).unsqueeze(0).repeat(self.n_layers, 1, 1)

        return outputs, dec_hidden

class BahdanauAttention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear(enc_hid_dim * 2 + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, dec_hidden, enc_outputs, src_mask):
        # dec_hidden: (num_layers, batch, dec_hid_dim)
        # enc_outputs: (batch, src_len, 2*enc_hid_dim)
        batch_size, src_len, _ = enc_outputs.shape
        dec_hidden_last = dec_hidden[-1].unsqueeze(1).repeat(1, src_len, 1)  # (batch, src_len, dec_hid_dim)

        energy = torch.tanh(
            self.attn(torch.cat((dec_hidden_last, enc_outputs), dim=2))
        )                                            # (batch, src_len, dec_hid_dim)
        scores = self.v(energy).squeeze(-1)         # (batch, src_len)
        scores = scores.masked_fill(src_mask, float("-inf"))
        attn_weights = F.softmax(scores, dim=1)     # (batch, src_len)
        return attn_weights

class DecoderGRUWithAttention(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim,
                 n_layers, dropout, pad_idx):
        super().__init__()
        self.output_dim = output_dim
        self.emb_dim = emb_dim
        self.dec_hid_dim = dec_hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=pad_idx)
        self.attention = BahdanauAttention(enc_hid_dim, dec_hid_dim)

        self.rnn = nn.GRU(
            emb_dim + enc_hid_dim * 2,
            dec_hid_dim,
            num_layers=n_layers,
            batch_first=True,
        )
        self.fc_out = nn.Linear(enc_hid_dim * 2 + dec_hid_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_tok, hidden, encoder_outputs, src_mask):
        # input_tok: (batch,)
        input_tok = input_tok.unsqueeze(1)          # (batch, 1)
        embedded = self.dropout(self.embedding(input_tok))  # (batch, 1, emb_dim)

        attn_weights = self.attention(hidden, encoder_outputs, src_mask)  # (batch, src_len)
        attn_weights = attn_weights.unsqueeze(1)                          # (batch, 1, src_len)
        context = torch.bmm(attn_weights, encoder_outputs)               # (batch, 1, 2*enc_hid_dim)

        rnn_input = torch.cat((embedded, context), dim=2)                # (batch, 1, emb+2*enc_hid)
        outputs, hidden = self.rnn(rnn_input, hidden)                    # outputs: (batch, 1, dec_hid)

        out_cat = torch.cat((outputs, context, embedded), dim=2)         # (batch, 1, dec_hid+2*enc_hid+emb)
        prediction = self.fc_out(out_cat).squeeze(1)                     # (batch, output_dim)

        return prediction, hidden, attn_weights.squeeze(1)


In [ ]:
class Seq2SeqBiGRU(nn.Module):
    def __init__(self, encoder, decoder, pad_idx):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_idx = pad_idx

    def create_src_mask(self, src):
        return (src == self.pad_idx)  # (batch, src_len)

    def forward(self, src, tgt, teacher_forcing_ratio=TEACHER_FORCING_RATIO):
        # src: (batch, src_len), tgt: (batch, tgt_len)
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, tgt_len, vocab_size, device=src.device)

        enc_outputs, hidden = self.encoder(src)
        src_mask = self.create_src_mask(src)

        input_tok = tgt[:, 0]  # BOS

        for t in range(1, tgt_len):
            output, hidden, _ = self.decoder(input_tok, hidden, enc_outputs, src_mask)
            outputs[:, t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_tok = tgt[:, t] if teacher_force else top1

        return outputs

encoder = EncoderBiGRU(
    input_dim=SRC_VOCAB_SIZE,
    emb_dim=EMB_SIZE,
    hid_dim=HID_SIZE,
    n_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_idx=PAD_IDX,
)

decoder = DecoderGRUWithAttention(
    output_dim=TGT_VOCAB_SIZE,
    emb_dim=EMB_SIZE,
    enc_hid_dim=HID_SIZE,
    dec_hid_dim=HID_SIZE,
    n_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_idx=PAD_IDX,
)

model_bigru = Seq2SeqBiGRU(encoder, decoder, PAD_IDX).to(device)

optimizer = torch.optim.Adam(model_bigru.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


In [ ]:
def train_epoch_bigru(model, loader):
    model.train()
    total_loss = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()

        outputs = model(src, tgt)
        vocab_size = outputs.size(-1)
        logits = outputs[:, 1:].contiguous().view(-1, vocab_size)
        tgt_out = tgt[:, 1:].contiguous().view(-1)

        loss = criterion(logits, tgt_out)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def eval_epoch_bigru(model, loader):
    model.eval()
    total_loss = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        outputs = model(src, tgt, teacher_forcing_ratio=0.0)
        vocab_size = outputs.size(-1)
        logits = outputs[:, 1:].contiguous().view(-1, vocab_size)
        tgt_out = tgt[:, 1:].contiguous().view(-1)
        loss = criterion(logits, tgt_out)
        total_loss += loss.item()
    return total_loss / len(loader)

EPOCHS = 15
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch_bigru(model_bigru, train_loader)
    val_loss = eval_epoch_bigru(model_bigru, val_loader)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")


Epoch 1: train_loss=5.7776, val_loss=4.8381
Epoch 2: train_loss=3.3362, val_loss=3.3806
Epoch 3: train_loss=1.8220, val_loss=2.4222
Epoch 4: train_loss=1.0622, val_loss=2.0044
Epoch 5: train_loss=0.6689, val_loss=1.8192
Epoch 6: train_loss=0.4246, val_loss=1.7518
Epoch 7: train_loss=0.2813, val_loss=1.5896
Epoch 8: train_loss=0.2057, val_loss=1.5044
Epoch 9: train_loss=0.1468, val_loss=1.4922
Epoch 10: train_loss=0.1108, val_loss=1.5633
Epoch 11: train_loss=0.0850, val_loss=1.4386
Epoch 12: train_loss=0.0742, val_loss=1.4469
Epoch 13: train_loss=0.0581, val_loss=1.5010
Epoch 14: train_loss=0.0566, val_loss=1.5699
Epoch 15: train_loss=0.0518, val_loss=1.4609


In [ ]:
EPOCHS = 10
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch_bigru(model_bigru, train_loader)
    val_loss = eval_epoch_bigru(model_bigru, val_loader)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

Epoch 1: train_loss=0.0453, val_loss=1.4542
Epoch 2: train_loss=0.0395, val_loss=1.5317
Epoch 3: train_loss=0.0317, val_loss=1.4669
Epoch 4: train_loss=0.0286, val_loss=1.4780
Epoch 5: train_loss=0.0246, val_loss=1.4810
Epoch 6: train_loss=0.0246, val_loss=1.5292
Epoch 7: train_loss=0.0340, val_loss=1.5431
Epoch 8: train_loss=0.0291, val_loss=1.5095
Epoch 9: train_loss=0.0188, val_loss=1.4746
Epoch 10: train_loss=0.0196, val_loss=1.5203


In [ ]:
@torch.no_grad()
def translate_bigru(text, max_len=MAX_LEN):
    model_bigru.eval()
    src_ids = [BOS_IDX] + sp_src.encode(text, out_type=int) + [EOS_IDX]
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)

    enc_outputs, hidden = model_bigru.encoder(src)
    src_mask = model_bigru.create_src_mask(src)

    input_tok = torch.tensor([BOS_IDX], device=device)
    generated = [BOS_IDX]

    for _ in range(max_len):
        output, hidden, _ = model_bigru.decoder(input_tok, hidden, enc_outputs, src_mask)
        top1 = output.argmax(1).item()
        if top1 == EOS_IDX:
            break
        generated.append(top1)
        input_tok = torch.tensor([top1], device=device)

    return sp_tgt.decode(generated[1:])

# quick sanity check
for i in range(5):
    src = final_dataset.iloc[i]["regional_text"]
    gold = final_dataset.iloc[i]["standard_bangla"]
    pred = translate_bigru(src)
    print("\nSRC :", src)
    print("GOLD:", gold)
    print("PRED:", pred)



SRC : ভালা আছনি?
GOLD: কেমন আছো ?
PRED: কেমন আছো ?

SRC : আইজকু আমার মন ভালা নায়
GOLD: আজকে আমার মন ভালো নেই
PRED: আজকে আমার মন ভালো নেই

SRC : তুমি কিতা খরো?
GOLD: তুমি কি করো ?
PRED: তুমি কি করো ?

SRC : অউ গরমো আমার কুনতা ভালা লাগের না
GOLD: এই গরমে আমার কিছু ভালো লাগে না
PRED: এই গরমে আমার কিছু ভালো লাগছে না

SRC : ফুয়াটায় এখটা সাদা রংগর শার্ট পিন্দিয়া আইছিল
GOLD: ছেলেটি সাদা রঙয়ের একটি শার্ট পরে এসেছিল
PRED: ছেলেটি সাদা রঙয়ের একটি শার্ট পরে এসেছিল


In [ ]:
!pip install sacrebleu rouge-score nltk sentence-transformers -q


import sacrebleu
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk import word_tokenize
import nltk
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt_tab', quiet=True)

st_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2").to(device)


@torch.no_grad()
def collect_predictions_bigru(model, df, max_samples=None):
    model.eval()
    if max_samples is not None and max_samples < len(df):
        eval_df = df.sample(n=max_samples, random_state=42)
    else:
        eval_df = df

    preds, refs = [], []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        src = str(row["regional_text"])
        tgt = str(row["standard_bangla"]).strip()
        pred = translate_bigru(src).strip()
        preds.append(pred)
        refs.append(tgt)
    return preds, refs
preds, refs = collect_predictions_bigru(model_bigru, val_df)
refs_list = [refs]


# BLEU (corpus)
bleu = sacrebleu.corpus_bleu(preds, refs_list).score

# ROUGE-L (corpus average)
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_scores = [scorer.score(r, p)['rougeL'].fmeasure for p, r in zip(preds, refs)]
rougeL = 100 * sum(rouge_scores) / len(rouge_scores)

# METEOR (sentence-level average)
meteor_scores = [
    meteor_score([word_tokenize(r)], word_tokenize(p))
    for p, r in zip(preds, refs)
]
meteor = 100 * sum(meteor_scores) / len(meteor_scores)

# Cosine similarity between sentence embeddings (source vs prediction)
# here we use reference vs prediction
emb_refs = st_model.encode(refs, convert_to_tensor=True, show_progress_bar=False)
emb_preds = st_model.encode(preds, convert_to_tensor=True, show_progress_bar=False)

cos_sims = cosine_similarity(
    emb_refs.cpu().numpy(),
    emb_preds.cpu().numpy()
)
# diagonal = each pair
cos_sim_vals = [cos_sims[i, i] for i in range(len(refs))]
cosine_sim = 100 * sum(cos_sim_vals) / len(cos_sim_vals)

print(f"Bi-GRU metrics on validation set:")
print(f"  BLEU        : {bleu:.2f}")
print(f"  ROUGE-L F1  : {rougeL:.2f}")
print(f"  METEOR      : {meteor:.2f}")
print(f"  Cosine Sim. : {cosine_sim:.2f}")

A = meteor       # adequacy
F = rougeL       # fluency
CS = cosine_sim  # semantic similarity

TQS = 0.4 * A + 0.3 * F + 0.3 * CS
print(f"TQS (0–100): {TQS:.2f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 938/938 [00:10<00:00, 88.25it/s]


Bi-GRU metrics on validation set:
  BLEU        : 59.14
  ROUGE-L F1  : 0.00
  METEOR      : 71.68
  Cosine Sim. : 94.60
TQS (0–100): 57.05


In [ ]:
for i in range(5):
    print("REF:", refs[i])
    print("PRED:", preds[i])
    print()


REF: এমন মনোভাব নিয়ে সংসারে কখনো সুখ আসে না
PRED: এমন মনোভাব নিয়ে সংসারে কখনো সুখ আসে না

REF: সে ২য় বিয়ে করছে একটা বিবাহিত মহিলাকে
PRED: সে ২য় বিয়ে করছে একটা বিবাহিত মহিলাকে

REF: আমার বিয়ের ২ মাসের মাথায় আব্বু মারা গেছেন
PRED: আমার বিয়ের ২ মাসের আমার বাবা মা মারা গেছেন

REF: আমাদের যৌথ পরিবার ছিল নানার বাসায়
PRED: আমাদের যৌথ পরিবার ছিল নানার বাসায়

REF: তুই কি এই গানটি শুনতে ভালোবাসবি?
PRED: তুই কি এই গানটি শুনতে ভালোবাসবি?



In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

scores = []
for p, r in zip(preds, refs):
    s = scorer.score(target=r, prediction=p)      # ref = target, pred = prediction
    scores.append(s['rougeL'].fmeasure)

print("First 5 ROUGE-L F1 values:", scores[:5])
print("Mean ROUGE-L F1 (0–1):", sum(scores)/len(scores))
print("Mean ROUGE-L F1 (0–100):", 100 * sum(scores)/len(scores))


First 5 ROUGE-L F1 values: [0, 0, 0, 0, 0]
Mean ROUGE-L F1 (0–1): 0.0
Mean ROUGE-L F1 (0–100): 0.0


In [ ]:
import re
import unicodedata
from rouge_score import rouge_scorer

def normalize_bangla(text: str) -> str:
    # 1) Unicode normalization
    text = unicodedata.normalize("NFC", text)

    # 2) Standardize common composed/decomposed forms (basic, you can extend)
    replacements = {
        "ি য়ে": "িয়ে",
        "ইয়ে": "িয়ে",
        "য়ে": "য়ে",
        "বিয়ে": "বিয়ে",
        "কোথোয়": "কোথায়",   # example – adapt to your orthography
    }
    for k, v in replacements.items():
        text = text.replace(k, v)

    # 3) Remove extra spaces and punctuation at edges
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_bangla(text: str) -> str:
    # Simple word-level tokenization: split by spaces, keep Bangla words intact
    # You can enhance this for punctuation if needed.
    text = normalize_bangla(text)
    # Optional: insert spaces around punctuation
    text = re.sub(r"([.,!?\"“”'’:;])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def compute_rouge_scores(refs, preds):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

    rouge1_f1 = []
    rougeL_f1 = []

    for p, r in zip(preds, refs):
        r_tok = tokenize_bangla(r)
        p_tok = tokenize_bangla(p)

        scores = scorer.score(target=r_tok, prediction=p_tok)
        rouge1_f1.append(scores['rouge1'].fmeasure)
        rougeL_f1.append(scores['rougeL'].fmeasure)

    rouge1 = sum(rouge1_f1) / len(rouge1_f1)
    rougeL = sum(rougeL_f1) / len(rougeL_f1)
    return rouge1, rougeL

rouge1, rougeL = compute_rouge_scores(refs, preds)

print(f"ROUGE-1 (0–1): {rouge1:.4f}")
print(f"ROUGE-L (0–1): {rougeL:.4f}")
print(f"ROUGE-1 (%): {rouge1*100:.2f}")
print(f"ROUGE-L (%): {rougeL*100:.2f}")


ROUGE-1 (0–1): 0.0000
ROUGE-L (0–1): 0.0000
ROUGE-1 (%): 0.00
ROUGE-L (%): 0.00


In [ ]:
!pip install rouge -q

from rouge import Rouge
import re
import unicodedata

rouge = Rouge()  # computes ROUGE-1, ROUGE-2, ROUGE-L

def normalize_bangla_basic(text: str) -> str:
    # NFC normalization and whitespace cleanup
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def bangla_word_tokenize(text: str) -> str:
    text = normalize_bangla_basic(text)
    # put spaces around punctuation
    text = re.sub(r"([.,!?\"“”'’:;()\[\]])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compute_rouge_with_rouge_pkg(refs, preds):
    # prepare lists of normalized, tokenized strings
    refs_tok = [bangla_word_tokenize(r) for r in refs]
    preds_tok = [bangla_word_tokenize(p) for p in preds]

    # rouge.get_scores works sentence-wise; we average
    scores = rouge.get_scores(preds_tok, refs_tok, avg=True)
    # scores is a dict with 'rouge-1', 'rouge-2', 'rouge-l'
    r1_f = scores["rouge-1"]["f"]   # 0–1
    rl_f = scores["rouge-l"]["f"]   # 0–1
    return r1_f, rl_f
r1, rl = compute_rouge_with_rouge_pkg(refs, preds)
print(f"ROUGE-1 (0–1): {r1:.4f}")
print(f"ROUGE-L (0–1): {rl:.4f}")
print(f"ROUGE-1 (%): {r1*100:.2f}")
print(f"ROUGE-L (%): {rl*100:.2f}")


ROUGE-1 (0–1): 0.8280
ROUGE-L (0–1): 0.8270
ROUGE-1 (%): 82.80
ROUGE-L (%): 82.70


In [ ]:
A  = meteor / 100.0        # e.g., 71.68 -> 0.7168
F  = 0.8270
CS = cosine_sim / 100.0    # e.g., 94.60 -> 0.9460

TQS = 0.4 * A + 0.3 * F + 0.3 * CS
print("TQS (0–1):", TQS)
print("TQS (%):", TQS * 100)


TQS (0–1): 0.81861264
TQS (%): 81.86127


In [ ]:
print(f"Bi-GRU metrics on validation set:")
print(f"  BLEU        : {bleu:.2f}")
print(f"ROUGE-1 (0–1): {r1:.4f}")
print(f"ROUGE-L (0–1): {rl:.4f}")
print(f"ROUGE-1 (%): {r1*100:.2f}")
print(f"ROUGE-L (%): {rl*100:.2f}")
print(f"  METEOR      : {meteor:.2f}")
print(f"  Cosine Sim. : {cosine_sim:.2f}")


print(f"TQS (0–100): {TQS:.2f}")

Bi-GRU metrics on validation set:
  BLEU        : 59.14
ROUGE-1 (0–1): 0.8280
ROUGE-L (0–1): 0.8270
ROUGE-1 (%): 82.80
ROUGE-L (%): 82.70
  METEOR      : 71.68
  Cosine Sim. : 94.60
TQS (0–100): 0.82


##**Comparison**

In [ ]:
!pip install sacrebleu rouge nltk sentence-transformers rouge-score -q

import sacrebleu
from rouge import Rouge
from nltk.translate.meteor_score import meteor_score
from nltk import word_tokenize
import nltk, re, unicodedata
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import torch

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

rouge = Rouge()
st_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2").to(device)

def normalize_bangla_basic(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def bangla_word_tokenize(text: str) -> str:
    text = normalize_bangla_basic(text)
    text = re.sub(r"([.,!?\"“”'’:;()\[\]])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


@torch.no_grad()
def evaluate_model(get_translation_func, df, max_samples=None):
    # ... (same as before up to preds, refs lists)

    preds, refs = [], []
    if max_samples is not None and max_samples < len(df):
        eval_df = df.sample(n=max_samples, random_state=42)
    else:
        eval_df = df

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        src = str(row["regional_text"])
        tgt = str(row["standard_bangla"]).strip()
        pred = get_translation_func(src).strip()

        preds.append(pred)
        refs.append(tgt)

    # BLEU / chrF (works fine even if some preds are empty)
    refs_list = [refs]
    bleu = sacrebleu.corpus_bleu(preds, refs_list).score
    chrf = sacrebleu.corpus_chrf(preds, refs_list).score

    # ---- ROUGE: avoid empty hypothesis ----
    refs_tok = []
    preds_tok = []
    for r, p in zip(refs, preds):
        r_t = bangla_word_tokenize(r)
        p_t = bangla_word_tokenize(p)
        if p_t == "":          # if model produced empty string
            p_t = "<empty>"    # minimal non-empty token
        refs_tok.append(r_t)
        preds_tok.append(p_t)

    r_scores = rouge.get_scores(preds_tok, refs_tok, avg=True)
    rouge1 = r_scores["rouge-1"]["f"]
    rougeL = r_scores["rouge-l"]["f"]

    # METEOR
    meteor_vals = [
        meteor_score([word_tokenize(r)], word_tokenize(p if p != "" else "<empty>"))
        for p, r in zip(preds, refs)
    ]
    meteor = sum(meteor_vals) / len(meteor_vals)

    # Cosine similarity
    emb_refs = st_model.encode(refs, convert_to_tensor=True, show_progress_bar=False)
    emb_preds = st_model.encode(
        [p if p != "" else "<empty>" for p in preds],
        convert_to_tensor=True,
        show_progress_bar=False
    )
    sims = cosine_similarity(emb_refs.cpu().numpy(), emb_preds.cpu().numpy())
    cos_vals = [sims[i, i] for i in range(len(refs))]
    cosine_sim = sum(cos_vals) / len(cos_vals)

    A  = meteor
    F  = rougeL
    CS = cosine_sim
    TQS = 0.4 * A + 0.3 * F + 0.3 * CS

    return {
        "BLEU": bleu,
        "chrF++": chrf,
        "ROUGE-1": rouge1,
        "ROUGE-L": rougeL,
        "METEOR": meteor,
        "CosineSim": cosine_sim,
        "TQS": TQS,
    }



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# 1) Bi-LSTM
bilstm_metrics = evaluate_model(
    lambda txt: translate_bilstm(txt, sp_src, sp_tgt),
    val_df
)

# 2) Bi-GRU
bigru_metrics = evaluate_model(
    lambda txt: translate_bigru(txt),   # ONLY txt
    val_df
)

# 3) Transformer seq2seq
# We pass 'model' explicitly to match the signature: translate_sentence(model, sentence)
transformer_metrics = evaluate_model(
    lambda txt: translate_sentence(model, txt),
    val_df
)

import pandas as pd

rows = [
    {"Model": "BiLSTM",      **bilstm_metrics},
    {"Model": "BiGRU",       **bigru_metrics},
    {"Model": "Transformer", **transformer_metrics},
]

df_metrics = pd.DataFrame(rows)

# Convert 0–1 metrics to %
for col in ["ROUGE-1", "ROUGE-L", "METEOR", "CosineSim", "TQS"]:
    df_metrics[col] = df_metrics[col] * 100

print(df_metrics.to_markdown(index=False, floatfmt=".2f"))

100%|██████████| 938/938 [00:26<00:00, 34.95it/s]


| Model       |   BLEU |   chrF++ |   ROUGE-1 |   ROUGE-L |   METEOR |   CosineSim |   TQS |
|:------------|-------:|---------:|----------:|----------:|---------:|------------:|------:|
| BiLSTM      |  60.55 |    74.93 |     82.67 |     82.63 |    71.76 |       94.17 | 81.74 |
| BiGRU       |  59.14 |    74.33 |     82.80 |     82.70 |    71.68 |       94.60 | 81.86 |
| Transformer |  38.25 |    65.78 |     63.50 |     63.44 |    57.60 |       92.40 | 69.79 |
